# 4.4 — Random Forests: Every Concept Explained From Scratch
**Deep Theory + Visuals + From-Scratch Code — Nothing Skipped**

## Table of Contents
1. Why Random Forest — the problem with single trees
2. Bagging — bootstrap aggregating from scratch
3. Out-of-Bag (OOB) error — free validation
4. Feature randomness — the 'random' in Random Forest
5. How prediction works — voting and averaging
6. n_estimators — how many trees do you need?
7. max_features — sqrt vs log2 vs all
8. Feature importance — MDI vs Permutation
9. Bias-Variance in Random Forest
10. Random Forest vs Bagging — the difference
11. Random Forest for Regression
12. Full Random Forest from scratch

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
np.random.seed(42)
print("Ready.")

---
## Concept 1 — Why Random Forest Exists

### The three problems with a single Decision Tree

**Problem 1: High Variance (Instability)**
Change a few training samples → completely different tree → completely different predictions.

**Problem 2: Overfitting**
A deep tree memorises noise. Pruning helps but creates a new problem:

**Problem 3: Pruned trees underfit**
Aggressively pruned tree → too simple → misses real patterns.

### The solution: Build many trees. Average them.

One unstable tree is wrong randomly. 500 trees are wrong in different random ways.
When you average random errors, they cancel out. The signal remains.

$$\text{Ensemble error} = \text{Bias}^2 + \frac{\text{Variance}}{n\_trees}$$

More trees → variance term shrinks → total error shrinks.

In [ ]:
# === Concept 1: Prove that averaging reduces error ===

np.random.seed(42)
X_base, y_base = make_moons(n_samples=600, noise=0.3, random_state=42)
X_base = StandardScaler().fit_transform(X_base)
X_btr, X_bte, y_btr, y_bte = train_test_split(X_base, y_base, test_size=0.3,
                                                 random_state=42, stratify=y_base)

print("=== Single Tree vs Ensemble ===")
# Run 20 single trees on different bootstrap samples and record test F1
single_f1s = []
for i in range(20):
    boot_idx = np.random.choice(len(X_btr), len(X_btr), replace=True)
    X_boot = X_btr[boot_idx]; y_boot = y_btr[boot_idx]
    dt = DecisionTreeClassifier(random_state=i)
    dt.fit(X_boot, y_boot)
    single_f1s.append(f1_score(y_bte, dt.predict(X_bte)))

# Random Forest at different n_estimators
rf_f1s = []
for n in range(1, 201):
    rf_n = RandomForestClassifier(n_estimators=n, random_state=42)
    rf_n.fit(X_btr, y_btr)
    rf_f1s.append(f1_score(y_bte, rf_n.predict(X_bte)))

plt.figure(figsize=(12, 5))
for i, f1 in enumerate(single_f1s):
    plt.axhline(f1, alpha=0.15, color='coral', linewidth=1)
plt.plot(range(1,201), rf_f1s, color='steelblue', linewidth=2.5, label='Random Forest F1')
plt.axhline(np.mean(single_f1s), color='coral', linewidth=2, linestyle='--',
            label=f'Avg single tree F1 = {np.mean(single_f1s):.3f}')
plt.xlabel('Number of Trees'); plt.ylabel('Test F1 Score')
plt.title('F1 Score vs Number of Trees\n'
          'Each faint red line = one single tree. Blue = ensemble of n trees.')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

print(f"Single tree F1 (average):   {np.mean(single_f1s):.4f}")
print(f"Single tree F1 (std):       {np.std(single_f1s):.4f}  ← high variance")
print(f"Random Forest (200 trees):  {rf_f1s[-1]:.4f}  ← stable, better")

---
## Concept 2 — Bagging: Bootstrap Aggregating

Bagging = the row-sampling part of Random Forest.

### Bootstrap sampling (sampling with replacement)
Each tree gets its own training set — same size as original, but sampled WITH replacement.

```
Original: [1,2,3,4,5,6,7,8,9,10]

Tree 1: [3,1,4,1,5,9,2,6,5,3]  ← 1 appears twice, 7/8/10 missing
Tree 2: [7,2,8,1,4,8,2,8,4,6]  ← 8 appears three times
Tree 3: [1,5,9,2,6,5,3,5,8,9]  ← different sample each time
```

### Why sampling WITH replacement?
- Creates diversity: each tree sees slightly different data → different errors
- Keeps each bootstrap sample the same size as original: n=800 → each tree still gets 800 rows

### What fraction of data does each tree see?
On average, each bootstrap sample contains about **63.2%** of unique original rows.
The remaining **36.8%** are never seen by that tree → these are Out-of-Bag rows.

$$P(\text{row not selected}) = \left(1-\frac{1}{n}\right)^n \rightarrow e^{-1} \approx 0.368$$

In [ ]:
# === Concept 2: Bootstrap sampling visualised ===

np.random.seed(42)
n_original = 20
original   = np.arange(1, n_original+1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, tree_num in enumerate([1,2,3,4]):
    sample = np.sort(np.random.choice(original, size=n_original, replace=True))
    oob    = sorted(set(original) - set(sample))
    counts = {i: list(sample).count(i) for i in original}

    colors = ['steelblue' if i in sample else 'lightgray' for i in original]
    bars   = ax.bar(original, [counts.get(i,0) for i in original], color=colors)

    ax.set_title(f'Tree {tree_num} bootstrap\n'
                 f'Unique seen: {len(set(sample))}/{n_original}\n'
                 f'OOB: {oob}', fontsize=8)
    ax.set_xlabel('Row ID'); ax.set_ylabel('Count (times sampled)')
    ax.axhline(1, color='gray', linestyle='--', alpha=0.5)
    ax.set_ylim(0, 4)

plt.suptitle('Bootstrap Sampling — Each Tree Gets Different Data\n'
             'Blue=sampled (may appear multiple times), Gray=OOB (never seen)', fontsize=11)
plt.tight_layout(); plt.show()

# Theoretical 63.2% calculation
print("=== Why each tree sees ~63.2% of unique rows ===")
n = 1000
theoretical = 1 - (1 - 1/n)**n
print(f"P(row IS selected at least once) = 1 - (1-1/n)^n")
print(f"  n=10:   {1-(1-1/10)**10:.4f}")
print(f"  n=100:  {1-(1-1/100)**100:.4f}")
print(f"  n=1000: {1-(1-1/1000)**1000:.4f}")
print(f"  n→∞:    1 - e^(-1) = {1-np.e**-1:.4f}")
print()

# Empirical verification
unique_fracs = []
for _ in range(1000):
    boot = np.random.choice(n, n, replace=True)
    unique_fracs.append(len(set(boot)) / n)
print(f"Empirical (1000 simulations): {np.mean(unique_fracs):.4f} ± {np.std(unique_fracs):.4f}")

---
## Concept 3 — Out-of-Bag (OOB) Error

OOB rows = the ~36.8% of training rows NOT seen by a tree.

**The clever trick:** use these as a free validation set.

For each training point $x_i$:
- Find all trees that did NOT see $x_i$ in their bootstrap sample
- Let only those trees vote on $x_i$
- Compare vote to true label $y_i$

Average error across all training points = OOB error.

**Why is this useful?**
- No separate validation set needed
- Unbiased estimate of generalisation error
- Built into sklearn: `oob_score=True`

In [ ]:
# === Concept 3: OOB error explained and verified ===

X_oob, y_oob = make_moons(n_samples=800, noise=0.25, random_state=42)
X_oob = StandardScaler().fit_transform(X_oob)
X_otr, X_ote, y_otr, y_ote = train_test_split(X_oob, y_oob, test_size=0.2,
                                                 random_state=42, stratify=y_oob)

rf_oob = RandomForestClassifier(
    n_estimators=200,
    oob_score=True,     # enable OOB evaluation
    random_state=42,
    n_jobs=-1
)
rf_oob.fit(X_otr, y_otr)

test_acc = rf_oob.score(X_ote, y_ote)
oob_acc  = rf_oob.oob_score_   # free, no test set needed

print("=== OOB Score as a Free Validation ===")
print(f"OOB accuracy (no test data used): {oob_acc*100:.2f}%")
print(f"Actual test accuracy:             {test_acc*100:.2f}%")
print(f"Difference:                       {abs(oob_acc-test_acc)*100:.2f}%")
print()
print("OOB score closely approximates test score.")
print("Use it when you want to conserve data (don't want a separate val set).")

# Show OOB decision function — per-sample probability from OOB trees only
oob_probs = rf_oob.oob_decision_function_   # shape (n_train, n_classes)
print(f"\nOOB decision function shape: {oob_probs.shape}")
print("Each row = probability from trees that did NOT train on that sample")
print(f"First 5 OOB probabilities (class 1): {oob_probs[:5,1].round(3)}")

---
## Concept 4 — Feature Randomness: The 'Random' in Random Forest

Bagging alone (bootstrapping rows) helps but doesn't fully solve the problem.

**The remaining problem:** if one feature is dominant (e.g. glucose in diabetes prediction), almost every tree will split on it first → trees are still correlated → averaging correlated trees doesn't help much.

### Solution: Random feature subset at each split

At every split of every tree, instead of considering ALL features, the algorithm picks a **random subset** of features and only tries those.

Default: `max_features = sqrt(n_features)`

For 16 features: only 4 randomly selected features are tried at each split.

**Effect:** Forces trees to be diverse. Even if glucose is dominant, trees that don't have it available at a given node must find the next-best feature → different trees, different paths.

### Why this works
- Individual trees: higher error (can't always use the best feature)
- Ensemble: errors are less correlated → average is much better
- Net result: better than bagging alone

In [ ]:
# === Concept 4: Feature randomness — prove it reduces correlation ===

np.random.seed(42)
X_fr, y_fr = make_classification(n_samples=1000, n_features=15, n_informative=5,
                                   n_redundant=5, random_state=42)
X_fr = StandardScaler().fit_transform(X_fr)
X_frtr, X_frte, y_frtr, y_frte = train_test_split(X_fr, y_fr, test_size=0.25,
                                                    random_state=42, stratify=y_fr)

# Measure prediction correlation between trees for different max_features
results_fr = {}
n_trees = 100

for label, mf in [('All features (Bagging)', 15),
                    ('sqrt features (RF)',    'sqrt'),
                    ('log2 features',         'log2'),
                    ('1 feature only',        1)]:
    rf_fr = RandomForestClassifier(n_estimators=n_trees, max_features=mf,
                                    random_state=42, n_jobs=-1)
    rf_fr.fit(X_frtr, y_frtr)

    # Collect individual tree predictions
    tree_preds = np.array([tree.predict(X_frte) for tree in rf_fr.estimators_])

    # Average pairwise correlation of predictions
    corr_matrix = np.corrcoef(tree_preds)
    avg_corr = corr_matrix[np.triu_indices_from(corr_matrix, k=1)].mean()

    ensemble_f1 = f1_score(y_frte, rf_fr.predict(X_frte))
    results_fr[label] = {'avg_tree_correlation': avg_corr, 'ensemble_f1': ensemble_f1}
    print(f"{label:<30}: tree correlation={avg_corr:.3f} | ensemble F1={ensemble_f1:.4f}")

print()
print("Lower tree correlation → more diverse trees → better ensemble performance")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels  = list(results_fr.keys())
corrs   = [results_fr[l]['avg_tree_correlation'] for l in labels]
f1s     = [results_fr[l]['ensemble_f1'] for l in labels]

axes[0].barh(labels, corrs, color='coral')
axes[0].set_title('Average Tree-to-Tree Correlation\n(Lower = more diverse)')
axes[0].set_xlabel('Correlation')

axes[1].barh(labels, f1s, color='steelblue')
axes[1].set_title('Ensemble F1 Score\n(Higher = better)')
axes[1].set_xlabel('F1 Score')

plt.tight_layout(); plt.show()

---
## Concept 5 — n_estimators: How Many Trees?

Adding more trees always reduces variance — but with diminishing returns.

### The law of diminishing returns
- Trees 1→10: massive improvement
- Trees 10→100: good improvement
- Trees 100→500: small improvement
- Trees 500→2000: barely any improvement

### Practical rule: 100-300 trees is enough for most datasets.

### Does more trees ever hurt?
No — more trees can never make things worse (only slower).
But past ~300, the improvement is negligible and training time grows linearly.

In [ ]:
# === Concept 5: n_estimators — law of diminishing returns ===

X_ne, y_ne = make_moons(n_samples=1000, noise=0.25, random_state=42)
X_ne = StandardScaler().fit_transform(X_ne)
X_netr, X_nete, y_netr, y_nete = train_test_split(X_ne, y_ne, test_size=0.2,
                                                    random_state=42, stratify=y_ne)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

n_tree_values = [1, 2, 3, 5, 7, 10, 15, 20, 30, 50, 75, 100, 150, 200, 300, 500]
test_f1s  = []
oob_scores= []
std_across_runs = []

for n in n_tree_values:
    # Run 5 times with different seeds to measure stability
    run_f1s = []
    for seed in range(5):
        rf_n = RandomForestClassifier(n_estimators=n, oob_score=(n>=5),
                                       random_state=seed, n_jobs=-1)
        rf_n.fit(X_netr, y_netr)
        run_f1s.append(f1_score(y_nete, rf_n.predict(X_nete)))
    test_f1s.append(np.mean(run_f1s))
    std_across_runs.append(np.std(run_f1s))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(n_tree_values, test_f1s, color='steelblue', marker='o', ms=5, linewidth=2)
axes[0].fill_between(n_tree_values,
                      np.array(test_f1s)-np.array(std_across_runs),
                      np.array(test_f1s)+np.array(std_across_runs),
                      alpha=0.2, color='steelblue', label='±1 std (stability)')
axes[0].set_xscale('log')
axes[0].set_title('F1 Score vs n_estimators (log scale)'); axes[0].set_xlabel('n_estimators')
axes[0].set_ylabel('F1 Score'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(n_tree_values, std_across_runs, color='coral', marker='o', ms=5, linewidth=2)
axes[1].set_xscale('log')
axes[1].set_title('Prediction Stability vs n_estimators\n'
                  '(lower std = same prediction regardless of random seed)')
axes[1].set_xlabel('n_estimators'); axes[1].set_ylabel('Std of F1 across 5 seeds')
axes[1].grid(True, alpha=0.3)

plt.suptitle('n_estimators — Law of Diminishing Returns', fontsize=12)
plt.tight_layout(); plt.show()

print("Key takeaways:")
print(f"  n=1:   std={std_across_runs[0]:.3f}  (very unstable — changes with every seed)")
print(f"  n=100: std={std_across_runs[n_tree_values.index(100)]:.3f}  (stable)")
print(f"  n=300: std={std_across_runs[n_tree_values.index(300)]:.3f}  (barely improves)")

---
## Concept 6 — Feature Importance: MDI vs Permutation

Random Forest gives two types of feature importance.

### Method 1: MDI (Mean Decrease in Impurity)
Built-in. Fast. Computed during training.

$$Importance_{MDI}(f) = \sum_{trees} \sum_{nodes\ using\ f} \frac{n_{node}}{n_{total}} \cdot \Delta Gini$$

**Problem:** Biased toward features with many unique values (high cardinality).
A feature with 1000 unique values gets more split opportunities → appears more important even if it isn't.

### Method 2: Permutation Importance
Slower but more reliable. Computed after training.

```
For each feature f:
    1. Shuffle column f in test data (break its relationship with target)
    2. Measure how much model accuracy drops
    3. Large drop → feature was important
    4. Small drop → feature was not contributing
```

**Advantage:** No bias toward cardinality. Works on test data → true generalisation importance.

In [ ]:
# === Concept 6: MDI vs Permutation Importance ===

from sklearn.datasets import load_diabetes
from sklearn.inspection import permutation_importance as perm_imp

# Use a dataset where one feature is high-cardinality noise
np.random.seed(42)
n = 800
# Real features
X_imp = np.column_stack([
    np.random.randn(n),           # feat 0: real signal
    np.random.randn(n)*2,         # feat 1: real signal (higher variance)
    np.random.randint(0, 500, n), # feat 2: HIGH CARDINALITY — 500 unique values, but NOISE
    np.random.randn(n)*0.1,       # feat 3: tiny signal
    np.arange(n),                 # feat 4: row ID — pure noise, very high cardinality
])
feat_names = ['Signal_1 (real)', 'Signal_2 (real)', 'HighCard_Noise (fake)',
              'Tiny_Signal', 'RowID_Noise (fake)']

y_imp = (X_imp[:,0] + X_imp[:,1] > 1).astype(int)  # only features 0 and 1 matter

X_itr, X_ite, y_itr, y_ite = train_test_split(X_imp, y_imp, test_size=0.25,
                                                 random_state=42, stratify=y_imp)

rf_imp = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_imp.fit(X_itr, y_itr)

# MDI importance
mdi = rf_imp.feature_importances_

# Permutation importance
perm = perm_imp(rf_imp, X_ite, y_ite, n_repeats=20, random_state=42, scoring='f1')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors_mdi = ['coral' if 'Noise' in n else 'steelblue' for n in feat_names]
axes[0].barh(feat_names, mdi, color=colors_mdi)
axes[0].set_title('MDI Importance (built-in)\n'
                  'RED = noise features. Should be near 0 but MDI overestimates them!')
axes[0].set_xlabel('Importance'); axes[0].invert_yaxis()

colors_perm = ['coral' if 'Noise' in n else 'steelblue' for n in feat_names]
axes[1].barh(feat_names, perm.importances_mean,
             xerr=perm.importances_std, color=colors_perm, capsize=4)
axes[1].set_title('Permutation Importance (reliable)\n'
                  'RED noise features correctly score near 0')
axes[1].set_xlabel('F1 drop when feature shuffled'); axes[1].invert_yaxis()

plt.suptitle('MDI vs Permutation Importance\n'
             'MDI is biased — high-cardinality noise gets inflated score', fontsize=11)
plt.tight_layout(); plt.show()

print("Takeaway:")
print("  MDI overstates high-cardinality features (HighCard_Noise, RowID_Noise)")
print("  Permutation correctly scores them near zero — they add no predictive value")
print("  Always use permutation importance when cardinality varies across features")

---
## Concept 7 — Bias-Variance in Random Forest

Individual deep trees: **low bias, high variance**  
Averaging many trees: **keeps low bias, dramatically reduces variance**

$$\text{Total Error} = \text{Bias}^2 + \frac{\rho \cdot \sigma^2 + (1-\rho)\sigma^2}{B}$$

Where:
- $\rho$ = correlation between trees (feature randomness reduces this)
- $\sigma^2$ = variance of individual tree
- $B$ = number of trees

As $B \to \infty$, the variance term approaches $\rho \cdot \sigma^2$ — never zero (because trees are still correlated).
Feature randomness reduces $\rho$ → reduces the floor of variance.

In [ ]:
# === Concept 7: Bias-Variance tradeoff in RF — visualised ===

np.random.seed(42)
# 1D regression setting — easy to visualise
X_bv = np.linspace(0, 8, 300).reshape(-1, 1)
y_true_bv = np.sin(X_bv).ravel()

n_datasets = 50
tree_preds  = []
rf10_preds  = []
rf100_preds = []

for ds in range(n_datasets):
    # Slightly different training data each time
    X_s = np.random.uniform(0, 8, 60).reshape(-1,1)
    y_s = np.sin(X_s).ravel() + np.random.randn(60)*0.4

    # Single deep tree
    dt_bv = DecisionTreeClassifier.__new__(DecisionTreeClassifier)
    from sklearn.tree import DecisionTreeRegressor
    dt_bv = DecisionTreeRegressor(random_state=ds)
    dt_bv.fit(X_s, y_s)
    tree_preds.append(dt_bv.predict(X_bv))

    # RF with 10 trees
    rf10 = RandomForestRegressor(n_estimators=10, random_state=ds, n_jobs=-1)
    rf10.fit(X_s, y_s); rf10_preds.append(rf10.predict(X_bv))

    # RF with 100 trees
    rf100 = RandomForestRegressor(n_estimators=100, random_state=ds, n_jobs=-1)
    rf100.fit(X_s, y_s); rf100_preds.append(rf100.predict(X_bv))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
configs = [
    (tree_preds,  'Single Decision Tree\n(Low bias, HIGH variance — wiggly)'),
    (rf10_preds,  'Random Forest (10 trees)\n(Lower variance, still some)'),
    (rf100_preds, 'Random Forest (100 trees)\n(Low bias, LOW variance — stable)'),
]

for ax, (preds, title) in zip(axes, configs):
    preds_arr = np.array(preds)
    mean_pred = preds_arr.mean(axis=0)
    std_pred  = preds_arr.std(axis=0)

    for p in preds_arr[:10]:
        ax.plot(X_bv, p, alpha=0.12, color='coral', linewidth=1)

    ax.plot(X_bv, y_true_bv, 'black', linewidth=2.5, label='True function (sin)')
    ax.plot(X_bv, mean_pred, 'steelblue', linewidth=2,   label='Mean prediction')
    ax.fill_between(X_bv.ravel(), mean_pred-std_pred, mean_pred+std_pred,
                    alpha=0.25, color='steelblue', label='±1 std (variance)')

    bias_sq  = np.mean((mean_pred - y_true_bv)**2)
    variance = np.mean(std_pred**2)
    ax.set_title(f'{title}\nBias²={bias_sq:.3f}  Variance={variance:.3f}', fontsize=9)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(-2, 2)

plt.suptitle('Bias-Variance Tradeoff — Single Tree vs Random Forest', fontsize=12)
plt.tight_layout(); plt.show()

---
## Concept 8 — Random Forest vs Bagging

| | Bagging | Random Forest |
|---|---|---|
| Row sampling | Bootstrap (with replacement) | Bootstrap (with replacement) |
| Feature sampling at each split | **All features** | **sqrt(n_features)** |
| Tree diversity | Medium | High |
| Tree correlation | Higher (dominant feature always wins) | Lower |
| Performance | Good | Better |

**The only difference:** feature randomness at each split.
But this small difference has a large impact because it de-correlates the trees.

In [ ]:
# === Concept 8: Bagging vs Random Forest — decision boundary comparison ===

X_brf, y_brf = make_moons(n_samples=600, noise=0.28, random_state=42)
X_brf = StandardScaler().fit_transform(X_brf)
X_btr, X_bte, y_btr, y_bte = train_test_split(X_brf, y_brf, test_size=0.25,
                                                 random_state=42, stratify=y_brf)

models = {
    'Single Decision Tree': DecisionTreeClassifier(random_state=42),
    'Bagging (all features)': BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=100, bootstrap=True, max_features=1.0,
        random_state=42, n_jobs=-1),
    'Random Forest (sqrt features)': RandomForestClassifier(
        n_estimators=100, max_features='sqrt', random_state=42, n_jobs=-1),
}

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
xx, yy = np.meshgrid(
    np.linspace(X_brf[:,0].min()-0.3, X_brf[:,0].max()+0.3, 250),
    np.linspace(X_brf[:,1].min()-0.3, X_brf[:,1].max()+0.3, 250)
)

for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_btr, y_btr)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    te_f1 = f1_score(y_bte, model.predict(X_bte))

    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
    ax.contour(xx, yy, Z, colors='gray', linewidths=0.6, alpha=0.5)
    ax.scatter(X_btr[y_btr==0,0], X_btr[y_btr==0,1], c='steelblue', s=20, alpha=0.7)
    ax.scatter(X_btr[y_btr==1,0], X_btr[y_btr==1,1], c='coral',     s=20, alpha=0.7)
    ax.set_title(f'{name}\nTest F1 = {te_f1:.4f}', fontsize=9)
    ax.grid(True, alpha=0.2)

plt.suptitle('Single Tree → Bagging → Random Forest\n'
             'Each step adds diversity → better, smoother boundary', fontsize=11)
plt.tight_layout(); plt.show()

---
## Concept 9 — Random Forest for Regression

Instead of majority vote → **average predictions** across all trees.

$$\hat{y} = \frac{1}{B} \sum_{b=1}^{B} T_b(x)$$

Where $T_b(x)$ is the prediction of tree $b$.

**Uncertainty estimation:** The spread (std) of predictions across trees = natural confidence interval.  
Wide std → trees disagree → model is uncertain about this region.

In [ ]:
# === Concept 9: RF Regression + uncertainty estimation ===

np.random.seed(42)
X_rfr = np.linspace(0, 10, 300).reshape(-1,1)
y_rfr = 2*np.sin(X_rfr).ravel() + np.random.randn(300)*0.5

X_rtr, X_rte, y_rtr, y_rte = train_test_split(X_rfr, y_rfr, test_size=0.3, random_state=42)

rf_reg = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_reg.fit(X_rtr.reshape(-1,1), y_rtr)

X_plot = np.linspace(0, 10, 500).reshape(-1, 1)

# Get individual tree predictions
individual_preds = np.array([tree.predict(X_plot) for tree in rf_reg.estimators_])
mean_pred = individual_preds.mean(axis=0)
std_pred  = individual_preds.std(axis=0)

plt.figure(figsize=(12, 5))
# Individual tree predictions (sample 20)
for p in individual_preds[:20]:
    plt.plot(X_plot, p, alpha=0.08, color='steelblue', linewidth=1)

plt.scatter(X_rtr, y_rtr, c='black', s=15, alpha=0.5, zorder=5, label='Training data')
plt.plot(X_plot, mean_pred, color='coral', linewidth=2.5, label='RF mean prediction')
plt.fill_between(X_plot.ravel(),
                 mean_pred - 2*std_pred,
                 mean_pred + 2*std_pred,
                 alpha=0.2, color='coral', label='±2 std (uncertainty)')

plt.title('RF Regression with Uncertainty Estimation\n'
          'Wider band = trees disagree = model is less certain in that region')
plt.xlabel('X'); plt.ylabel('y')
plt.legend(fontsize=9); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Numeric summary
from sklearn.metrics import mean_squared_error, r2_score
y_hat_test = rf_reg.predict(X_rte.reshape(-1,1))
print(f"RF Regression on test set:")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_rte, y_hat_test)):.4f}")
print(f"  R²:   {r2_score(y_rte, y_hat_test):.4f}")
print(f"\nAverage prediction uncertainty (std): {std_pred.mean():.4f}")
print("Use std per prediction to flag low-confidence predictions for manual review.")

---
## Concept 10 — Full Random Forest from Scratch

A minimal but complete implementation.

In [ ]:
# === Concept 10: Random Forest from scratch ===

from collections import Counter

class RandomForestScratch:
    """
    Random Forest Classifier from scratch.
    Implements: bootstrap sampling + feature randomness + majority vote.
    """

    def __init__(self, n_estimators=100, max_depth=None,
                 max_features='sqrt', min_samples_leaf=1, random_state=None):
        self.n_estimators    = n_estimators
        self.max_depth       = max_depth
        self.max_features    = max_features
        self.min_samples_leaf= min_samples_leaf
        self.random_state    = random_state
        self.trees           = []    # list of (DecisionTreeClassifier, feature_indices)

    def _get_n_features(self, total_features):
        if self.max_features == 'sqrt':
            return max(1, int(np.sqrt(total_features)))
        elif self.max_features == 'log2':
            return max(1, int(np.log2(total_features)))
        elif isinstance(self.max_features, int):
            return self.max_features
        else:
            return total_features

    def fit(self, X, y):
        X, y = np.array(X, float), np.array(y)
        n_samples, n_features = X.shape
        rng = np.random.RandomState(self.random_state)

        n_feat_per_tree = self._get_n_features(n_features)
        self.trees = []

        for i in range(self.n_estimators):
            # Step 1: Bootstrap sample (rows)
            boot_idx = rng.choice(n_samples, n_samples, replace=True)
            X_boot   = X[boot_idx]
            y_boot   = y[boot_idx]

            # Step 2: Random feature subset
            feat_idx = rng.choice(n_features, n_feat_per_tree, replace=False)
            X_sub    = X_boot[:, feat_idx]

            # Step 3: Train a full decision tree on this bootstrap + feature subset
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf,
                random_state=rng.randint(0, 10000)
            )
            tree.fit(X_sub, y_boot)
            self.trees.append((tree, feat_idx))

        return self

    def predict(self, X):
        X = np.array(X, float)
        # Collect votes from all trees
        all_votes = np.array([
            tree.predict(X[:, feat_idx])
            for tree, feat_idx in self.trees
        ])   # shape: (n_trees, n_samples)

        # Majority vote per sample
        return np.array([
            Counter(all_votes[:, i]).most_common(1)[0][0]
            for i in range(X.shape[0])
        ])

    def predict_proba(self, X):
        X = np.array(X, float)
        classes = np.unique(np.concatenate([
            tree.classes_ for tree, _ in self.trees
        ]))
        proba = np.zeros((X.shape[0], len(classes)))
        for tree, feat_idx in self.trees:
            tp = tree.predict_proba(X[:, feat_idx])
            for j, c in enumerate(tree.classes_):
                ci = np.where(classes == c)[0][0]
                proba[:, ci] += tp[:, j]
        return proba / self.n_estimators

    def score(self, X, y):
        return (self.predict(X) == np.array(y)).mean()


# ── Test vs sklearn ──
X_fs, y_fs = make_moons(n_samples=600, noise=0.25, random_state=42)
X_fs = StandardScaler().fit_transform(X_fs)
X_fstr, X_fste, y_fstr, y_fste = train_test_split(X_fs, y_fs, test_size=0.25,
                                                    random_state=42, stratify=y_fs)

our_rf = RandomForestScratch(n_estimators=50, max_features='sqrt',
                               max_depth=10, random_state=42)
our_rf.fit(X_fstr, y_fstr)

sk_rf = RandomForestClassifier(n_estimators=50, max_features='sqrt',
                                max_depth=10, random_state=42, n_jobs=-1)
sk_rf.fit(X_fstr, y_fstr)

print("=== From Scratch vs sklearn ===")
print(f"Our RF accuracy:     {our_rf.score(X_fste, y_fste)*100:.2f}%")
print(f"sklearn RF accuracy: {sk_rf.score(X_fste, y_fste)*100:.2f}%")
print(f"Our RF F1:           {f1_score(y_fste, our_rf.predict(X_fste)):.4f}")
print(f"sklearn RF F1:       {f1_score(y_fste, sk_rf.predict(X_fste)):.4f}")
print("Small differences are expected (sklearn has additional optimisations)")

---
## Summary — Every Random Forest Concept at a Glance

| Concept | Key point |
|---|---|
| Why RF exists | Single trees: high variance, unstable. RF averages them away. |
| Bagging | Each tree trains on a bootstrap sample (rows with replacement). |
| OOB rows | ~36.8% of rows not seen by each tree → free validation set. |
| OOB score | `oob_score=True` gives unbiased test accuracy for free. |
| Feature randomness | Each split considers only sqrt(n_features) random features. |
| Effect of randomness | De-correlates trees → averaging helps more → better ensemble. |
| n_estimators | More trees = lower variance. Diminishing returns after ~100-300. |
| max_features | `sqrt` (default) → classification. `1/3` often good for regression. |
| MDI importance | Fast. Built-in. Biased toward high-cardinality features. |
| Permutation importance | Slow. No bias. Reliable. Use this for feature selection. |
| Bias-Variance | Individual trees: low bias, high variance. RF: low bias, low variance. |
| Regression | Average predictions across trees. Std = natural uncertainty estimate. |
| No scaling needed | Tree-based. Uses thresholds not distances. |

### When to use Random Forest
- General-purpose first choice for tabular data
- Mixed feature types (no encoding or scaling needed)
- Need feature importance
- When you can't afford the instability of a single tree
- Medium-to-large datasets

### When NOT to use
- Very large datasets (slow, memory intensive)
- Need human-readable rules (use single Decision Tree)
- Need probability calibration (RF probabilities are not well-calibrated)
- Real-time prediction on edge devices (model size too large)

### The one line to remember
**Diversity × Quantity = Power.**  
Each tree is slightly wrong in its own unique way.  
500 unique mistakes → average → one reliable prediction.